[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Peewee, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)

# SQLite and PostgreSQL


## What you will be able to do

Keep one models module and change the database under it, with `bind_ctx` and a URL. Read the SQL
peewee would send to each backend without connecting to either, and say what changes: the key
column, the placeholder, the `RETURNING` clause, and the functions that do not exist on the other
side. Build a `Database` from a URL, and say what that call does and does not do. Produce
`database is locked` on purpose, and say what WAL buys and what it does not. Say what a full text
search costs when the backend changes, with both failures shown in the order they arrive. Use a
connection pool, and know what has to happen to it after a fork.


## The idea

### The problem

Everything in this guide has run on SQLite, which is a file. Most of it will run unchanged on
PostgreSQL, which is a server, and some of it will not. Knowing which is which before the move is
worth more than any amount of care afterwards.

peewee makes the move small on purpose: the models do not name a backend, the query builder compiles
to whichever dialect the database in scope wants, and the `Database` object is the only thing that
has to change. What does not move is anything that reached past the builder into one backend's own
features, which in this guide means the FTS5 search and, in most applications, means one or two
hand-written statements nobody remembers writing.

### What the switch is

A different `Database` object. `playhouse.db_url.connect` builds one from a URL, so the backend
becomes a setting rather than an import. `bind_ctx` points existing model classes at a database for
the length of a block, which is how the same models get tested against two.

### Why it works that way

A query is not SQL until something compiles it, and what compiles it is the database in scope. That
is why `query.sql()` needs a database to answer, and why it can answer without a connection: dialect
is a property of the class, not of the socket. So the whole contrast in this notebook is printed
from a `PostgresqlDatabase` that has never spoken to a server.

### Where this shows up

The move from a prototype to something with more than one process writing to it, which is the point
where SQLite's single writer starts to be the constraint rather than a detail. Also any project
that develops on SQLite and deploys on PostgreSQL, which is common and is where the surprises are
found late.

### What this notebook covers

The same models compiled for both backends, side by side. A database from a URL. `database is
locked`, produced on purpose, and what WAL changes about it. The search that does not port, in the
two stages it fails in. What the switch buys, each item pointing back at the notebook that showed
the SQLite side. A connection pool. Then the four failures.

No PostgreSQL server is running here, and the notebook does not pretend otherwise: everything about
PostgreSQL below is compiled rather than executed, and the one cell that tries to connect is there
to show what that failure looks like.

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
from peewee import CharField, Model, PostgresqlDatabase, SqliteDatabase, TextField

sqlite = SqliteDatabase(None)                   # no file named, and no connection opened
postgres = PostgresqlDatabase(None)


class Note(Model):
    title = CharField(max_length=80)
    body = TextField()

    class Meta:
        database = sqlite


for name, database in (("sqlite", sqlite), ("postgres", postgres)):
    with database.bind_ctx([Note]):             # the same class, compiled for another backend
        created = Note._schema._create_table().query()[0]
        insert = " ".join(Note.insert(title="x", body="y").sql()[0].split())
        print(f"{name:<9} key column: {created.split('(')[1].split(',')[0]}")
        print(f"{'':<9} insert:     {insert}")
```

```
sqlite    key column: "id" INTEGER NOT NULL PRIMARY KEY
          insert:     INSERT INTO "note" ("title", "body") VALUES (?, ?)
postgres  key column: "id" SERIAL NOT NULL PRIMARY KEY
          insert:     INSERT INTO "note" ("title", "body") VALUES (%s, %s) RETURNING "note"."id"
```

One class, two dialects, and no database was opened to produce either. The key column changed type,
the placeholder changed shape, and PostgreSQL's insert grew a `RETURNING` clause, which is how
peewee gets the new key back there instead of asking the cursor for it.


## Setup

Eleven imports, peewee installed and pinned, one model, and four helpers.

- `peewee` is the library, and `Model`, the field classes, `SqliteDatabase` and `PostgresqlDatabase`,
  from it, are what a model and a backend are written with
- `connect_url`, from `playhouse.db_url`, builds a `Database` from a URL string
- `PooledSqliteDatabase`, from `playhouse.pool`, is the pool one section runs
- `TSVectorField`, from `playhouse.postgres_ext`, is the PostgreSQL search field that does not port
- `tempfile` and `Path` make the one directory that needs real files, and `re` trims a connection
  error down to the part that is the same on every operating system
- `subprocess` and `sys` install peewee 4.5.1 where the version is not that, with `version` and
  `PackageNotFoundError`

`sqlite` and `postgres` are both `Database` objects built with `None`, which names no file and no
server. That is enough to compile SQL and not enough to run it, which is exactly what this notebook
wants: `compiled` and `created` print what would be sent, under either backend, with nothing open.


In [1]:
import re
import tempfile
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    if version("peewee") != "4.5.1":                                # Colab has 4.4.0, whose wording differs
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "peewee==4.5.1"], check=True)

import peewee
from peewee import (CharField, IntegerField, Model, OperationalError, PostgresqlDatabase,
                    SqliteDatabase, TextField, fn)
from playhouse.db_url import connect as connect_url
from playhouse.pool import PooledSqliteDatabase
from playhouse.postgres_ext import TSVectorField

WORK = Path(tempfile.mkdtemp(prefix="backends-"))                   # for the one section that needs files

sqlite = SqliteDatabase(None)                                       # named later, or never
postgres = PostgresqlDatabase(None)


class Note(Model):
    """One models module, for both backends."""

    title = CharField(max_length=80)
    body = TextField()
    reads = IntegerField(default=0)

    class Meta:
        database = sqlite


def compiled(build, database):
    """The SQL a query would send under a database, with nothing connected.

    `build` is a function rather than a query, because a query remembers the database it was
    built against: building it outside this block would compile it for the wrong backend.
    """
    with database.bind_ctx([Note]):
        return " ".join(build().sql()[0].split())


def created(database):
    """The CREATE TABLE peewee would emit for Note under a database."""
    with database.bind_ctx([Note]):
        return Note._schema._create_table().query()[0]


def refused(message):
    """A connection failure without the part of it that differs between operating systems."""
    return re.sub(r"(port \d+ failed):.*", r"\1: ...", message.strip().splitlines()[0])


print("peewee", peewee.__version__)
print("nothing is connected:", sqlite.is_closed(), postgres.is_closed())


peewee 4.5.1
nothing is connected: True True


## Worked examples

### The same model, compiled twice

`bind_ctx` points the class at another database for the length of a block and puts it back
afterwards:


In [2]:
for name, database in (("sqlite", sqlite), ("postgres", postgres)):
    print(f"{name}:")
    print("  ", created(database))

print()
print("bound to, afterwards:", type(Note._meta.database).__name__)


sqlite:
   CREATE TABLE IF NOT EXISTS "note" ("id" INTEGER NOT NULL PRIMARY KEY, "title" VARCHAR(80) NOT NULL, "body" TEXT NOT NULL, "reads" INTEGER NOT NULL)
postgres:
   CREATE TABLE IF NOT EXISTS "note" ("id" SERIAL NOT NULL PRIMARY KEY, "title" VARCHAR(80) NOT NULL, "body" TEXT NOT NULL, "reads" INTEGER NOT NULL)

bound to, afterwards: SqliteDatabase


`INTEGER PRIMARY KEY` against `SERIAL PRIMARY KEY`, and `VARCHAR(80)` the same in both. The model
said `CharField(max_length=80)` and each backend was told what that means in its own terms.

### What changes in a query

The four differences worth knowing, all visible without a server. Each query is built inside the
block rather than before it, because a query remembers the database it was built against:


In [3]:
builders = {
    "insert": lambda: Note.insert(title="x", body="y"),
    "select": lambda: Note.select(Note.title).where(Note.title == "x").limit(2),
    "update": lambda: Note.update({Note.reads: Note.reads + 1}).where(Note.title == "x"),
}

for label, build in builders.items():
    print(f"{label}:")
    for name, database in (("sqlite  ", sqlite), ("postgres", postgres)):
        print(f"  {name} {compiled(build, database)}")


insert:
  sqlite   INSERT INTO "note" ("title", "body", "reads") VALUES (?, ?, ?)
  postgres INSERT INTO "note" ("title", "body", "reads") VALUES (%s, %s, %s) RETURNING "note"."id"
select:
  sqlite   SELECT "t1"."title" FROM "note" AS "t1" WHERE ("t1"."title" = ?) LIMIT ?
  postgres SELECT "t1"."title" FROM "note" AS "t1" WHERE ("t1"."title" = %s) LIMIT %s
update:
  sqlite   UPDATE "note" SET "reads" = ("note"."reads" + ?) WHERE ("note"."title" = ?)
  postgres UPDATE "note" SET "reads" = ("note"."reads" + %s) WHERE ("note"."title" = %s)


| What | SQLite | PostgreSQL |
|---|---|---|
| the key column | `INTEGER PRIMARY KEY` | `SERIAL PRIMARY KEY` |
| the placeholder | `?` | `%s` |
| the new key after an insert | read from the cursor | a `RETURNING` clause in the statement |
| a function only one side has | compiles anyway | compiles anyway |

The first three are not things your code says. They are what the same code becomes. The fourth is
the one to watch, because compiling is not checking:


In [4]:
for name, database in (("sqlite  ", sqlite), ("postgres", postgres)):
    matched = compiled(lambda: Note.select(Note.title).where(fn.MATCH(Note.body, "sea")), database)
    print(f"  {name} {matched.split('WHERE ')[1]}")


  sqlite   MATCH("t1"."body", ?)
  postgres MATCH("t1"."body", %s)


`MATCH` is SQLite's. peewee compiled it for PostgreSQL too, because `fn.ANYTHING` is how you name a
function the builder has never heard of, and the builder is in no position to know which functions a
server has. That second line is SQL no PostgreSQL parser accepts: it would go over the wire and come
back as an error about a function that does not exist.

The rule is that the builder guarantees dialect, not vocabulary. Anything written through `fn`, or
through `raw` and `execute_sql` from **Selecting Rows**, is yours to check.

### A database from a URL

`connect_url` reads the scheme and builds the matching `Database`:


In [5]:
for url in (f"sqlite:///{WORK / 'catalog.db'}", "postgresql://someone@127.0.0.1:59999/catalog"):
    database = connect_url(url)
    print(f"  {url.split('://')[0]:<11} -> {type(database).__name__}, closed: {database.is_closed()}")


  sqlite      -> SqliteDatabase, closed: True
  postgresql  -> PostgresqlDatabase, closed: True


Read that second line carefully. There is no PostgreSQL server on port 59999, and `connect_url`
returned an object anyway. The name is misleading: it builds a database from a URL and does not
connect to anything. Connecting is a separate call, and it is the one that fails, which is the
second of the Common errors below.

What this buys is that the backend becomes a string in a settings file:


In [6]:
def database_for(url):
    """The Database an application would use, chosen by a setting rather than by an import."""
    return connect_url(url)


for url in (f"sqlite:///{WORK / 'catalog.db'}", "postgresql://someone@127.0.0.1:59999/catalog"):
    chosen = database_for(url)
    with chosen.bind_ctx([Note]):
        print(f"  {type(chosen).__name__:<20} {compiled(lambda: Note.insert(title='x', body='y'), chosen)}")


  SqliteDatabase       INSERT INTO "note" ("title", "body", "reads") VALUES (?, ?, ?)
  PostgresqlDatabase   INSERT INTO "note" ("title", "body", "reads") VALUES (%s, %s, %s) RETURNING "note"."id"


### database is locked

SQLite allows one writer at a time. A second writer waits, and if it waits longer than the timeout
it gives up with an error rather than blocking forever. The default wait is five seconds, which is
long enough to hide the problem in development, so here it is set to one:


In [7]:
path = str(WORK / "locked.db")
maker = SqliteDatabase(path)
with maker.bind_ctx([Note]):                                        # Note is bound to nothing yet
    maker.create_tables([Note])

first = SqliteDatabase(path, timeout=1)
second = SqliteDatabase(path, timeout=1)
first.connect()
second.connect()

first.begin()                                                       # a write transaction, left open
first.execute_sql("INSERT INTO note (title, body, reads) VALUES (?, ?, ?)", ("one", "b", 0))

try:
    second.execute_sql("INSERT INTO note (title, body, reads) VALUES (?, ?, ?)", ("two", "b", 0))
    print("the second writer got in")
except OperationalError as error:
    print("peewee.OperationalError:", error)

first.rollback()
first.close()
second.close()


peewee.OperationalError: database is locked


True

The timeout is what turned a wait into an error. With the default five seconds and a transaction
that ends quickly, the same two writers would have taken turns and nobody would have noticed. That
is the honest shape of this problem: it is a queue until the queue is longer than the timeout, and
then it is an exception, under load, in production.

SQLite's locking model itself belongs to **sqlite3, Deep Dive**. What matters here is what it costs
and what can be done about it from peewee.

### What WAL does and does not buy


In [8]:
wal_path = str(WORK / "wal.db")
writer = SqliteDatabase(wal_path, timeout=1, pragmas={"journal_mode": "wal"})
reader = SqliteDatabase(wal_path, timeout=1, pragmas={"journal_mode": "wal"})
writer.connect()
with writer.bind_ctx([Note]):
    writer.create_tables([Note])
reader.connect()

writer.begin()
writer.execute_sql("INSERT INTO note (title, body, reads) VALUES (?, ?, ?)", ("one", "b", 0))

print("a reader, while a write is open:", reader.execute_sql("SELECT COUNT(*) FROM note").fetchone())
try:
    reader.execute_sql("INSERT INTO note (title, body, reads) VALUES (?, ?, ?)", ("two", "b", 0))
    print("a second writer under wal: allowed")
except OperationalError as error:
    print("a second writer under wal:", error)

writer.rollback()
writer.close()
reader.close()


a reader, while a write is open: (0,)
a second writer under wal: database is locked


True

That is the whole trade, and it is smaller than it is usually described as being. Under the default
journal a reader can be blocked by a writer. Under WAL it is not: the read went through while a
write transaction was open. What WAL does not do is allow two writers, and the second one is locked
out exactly as before.

So WAL is worth setting on any SQLite application that reads while it writes, and it is not the
thing that makes SQLite handle concurrent writes, because nothing does.

### The search that does not port

The **FTS5Model and SearchField** notebook built a search that is SQLite's own. PostgreSQL's
equivalent is a `tsvector` column, and peewee has a field for it. Pointed at SQLite it fails twice,
in this order:


In [9]:
class Indexed(Model):
    body = TextField()
    search = TSVectorField()                                        # PostgreSQL's full text column

    class Meta:
        database = sqlite


on_file = SqliteDatabase(str(WORK / "search.db"))
with on_file.bind_ctx([Indexed]):
    print("the table compiles:", Indexed._schema._create_table().query()[0][:74], "...")
    for statement in Indexed._schema._create_indexes():
        print("the index it wants:", statement.query()[0])
    try:
        on_file.create_tables([Indexed])
    except OperationalError as error:
        print("creating it:      peewee.OperationalError:", error)


the table compiles: CREATE TABLE IF NOT EXISTS "indexed" ("id" INTEGER NOT NULL PRIMARY KEY, " ...
the index it wants: CREATE INDEX IF NOT EXISTS "indexed_search" ON "indexed" USING GIN ("search")
creating it:      peewee.OperationalError: near "USING": syntax error


The table itself is fine, because SQLite accepts a column type it has never heard of and gives it an
affinity. The index is not: `USING GIN` is PostgreSQL's, and there is nothing in SQLite it could
mean.

Take the index away and it gets further, which is worse:


In [10]:
class Loose(Model):
    body = TextField()
    search = TSVectorField(index=False)                             # no GIN index this time

    class Meta:
        database = sqlite


with on_file.bind_ctx([Loose]):
    on_file.create_tables([Loose])
    Loose.create(body="the sea", search="sea")
    print("the table was made and a row written:", Loose.select().count())

    query = Loose.select().where(Loose.search.match("sea"))
    print("the filter compiles to:", " ".join(query.sql()[0].split()).split("WHERE ")[1])
    try:
        print([row.body for row in query])
    except OperationalError as error:
        print("running it:   peewee.OperationalError:", error)


the table was made and a row written: 1
the filter compiles to: ("t1"."search" @@ to_tsquery(?))
running it:   peewee.OperationalError: unrecognized token: "@"


`@@` is PostgreSQL's match operator and SQLite does not have an `@` at all. So the column was
created, rows went in, and the failure waited for the first search, which is the worst place for it.

### What the switch buys

Each of these is something an earlier notebook showed the SQLite side of, described here rather than
demonstrated, because demonstrating them needs a server:

- **Foreign keys enforced, with nothing to switch on.** The **Relationships** notebook had to build
  every database with `pragmas={"foreign_keys": 1}`. PostgreSQL has no such setting.
- **A failed statement aborts the transaction.** The **Transactions** notebook's swallowed
  `IntegrityError` quietly commits half a batch on SQLite. On PostgreSQL every later statement in
  that block fails with `current transaction is aborted, commands ignored until end of transaction
  block`, so the savepoint per risky write stops being tidiness and becomes necessary.
- **`on_conflict` needs a `conflict_target`.** The **Creating and Changing Rows** notebook used one
  anyway, which ports. SQLite's `on_conflict_replace` does not.
- **Full text search does not port at all.** FTS5 is a virtual table. The PostgreSQL twin is a
  `tsvector` column with a GIN index, searched with `TSVectorField.match(query, websearch=True)`.
- **Connection pooling becomes the normal thing.** A server connection is expensive where a file
  handle is not.

### Connections from a pool

`playhouse.pool` has a pooled version of each backend. The SQLite one is real code you can run, and
behaves the same way the PostgreSQL one does:


In [11]:
pool = PooledSqliteDatabase(str(WORK / "pool.db"), max_connections=4, stale_timeout=30)

with pool.bind_ctx([Note]):
    pool.connect()
    pool.create_tables([Note])
    Note.create(title="pooled", body="b")
    print("while it is open  -> in use:", len(pool._in_use), "| idle:", len(pool._connections))
    pool.close()
    print("after close       -> in use:", len(pool._in_use), "| idle:", len(pool._connections))
    pool.connect()
    pool.close()
    print("connected again   -> idle:", len(pool._connections), "(the same one, reused)")
    pool.dispose()
    print("after dispose     -> idle:", len(pool._connections))


while it is open  -> in use: 1 | idle: 0
after close       -> in use: 0 | idle: 1
connected again   -> idle: 1 (the same one, reused)
after dispose     -> idle: 0


`close` gives the connection back to the pool rather than closing it, which is the point. `dispose`
throws the pool away, and it is what a child process has to call after a fork: the parent's
connections were copied into it, two processes cannot use one socket, and the symptoms of not doing
this are some of the strangest a database will ever give you.

### When to reach for which

| What you want | How to write it |
|---|---|
| the backend as a setting | `connect_url(os.environ["DATABASE_URL"])` |
| the same models against another backend | `with other.bind_ctx([Model, ...]):` |
| to see what a backend would be sent | `query.sql()` inside a `bind_ctx` |
| a SQLite reader not blocked by a writer | `pragmas={"journal_mode": "wal"}` |
| a SQLite writer to wait longer | `timeout=<seconds>` |
| connections reused rather than remade | `PooledPostgresqlDatabase`, or the SQLite one |
| a usable database in a forked child | `db.dispose()` in the child, first thing |
| full text search on PostgreSQL | `TSVectorField` and `match(..., websearch=True)` |

The default is to write nothing backend specific and let the builder compile it. Where that is not
possible, which is mostly search, expect to write both and choose at the seam rather than hoping.

### One settings line, finished

An application whose backend is a string, with the SQL it would send printed for each, and nothing
connected.


In [12]:
def show_plan(url):
    """What an application configured with this URL would send, without connecting to it."""
    database = connect_url(url)
    with database.bind_ctx([Note]):
        write = lambda: Note.insert(title="Tides", body="the sea", reads=0)
        read = lambda: Note.select(Note.title).where(Note.reads > 0).limit(5)
        return type(database).__name__, compiled(write, database), compiled(read, database)


for url in (f"sqlite:///{WORK / 'catalog.db'}", "postgresql://user@db.internal:5432/catalog"):
    backend, write, read = show_plan(url)
    print(f"{backend}:")
    print("   write:", write)
    print("   read: ", read)


SqliteDatabase:
   write: INSERT INTO "note" ("title", "body", "reads") VALUES (?, ?, ?)
   read:  SELECT "t1"."title" FROM "note" AS "t1" WHERE ("t1"."reads" > ?) LIMIT ?
PostgresqlDatabase:
   write: INSERT INTO "note" ("title", "body", "reads") VALUES (%s, %s, %s) RETURNING "note"."id"
   read:  SELECT "t1"."title" FROM "note" AS "t1" WHERE ("t1"."reads" > %s) LIMIT %s


The only difference between those two runs is the string at the top. Everything else, including the
model, the queries and this function, is the same code.

### Where each part came from

| In the plan | What it relies on | The section that showed it |
|---|---|---|
| `connect_url(url)` | a backend chosen by a setting | A database from a URL |
| `database.bind_ctx([Note])` | the same class against another backend | The same model, compiled twice |
| `query.sql()` with nothing open | dialect belonging to the class | The idea |
| the `%s` and the `RETURNING` | what the switch changes | What changes in a query |
| no connection at any point | a `Database` built with `None` | Setup |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/peewee-deep-dive/11-sqlite-and-postgresql-solutions.ipynb).

**1.** Print the `CREATE TABLE` for `Note` under both backends and name every difference between
them.


In [13]:
# your code here


**2.** Print the SQL for a `delete` and for a `select` with `limit` under both backends, and say
what changed in each.


In [14]:
# your code here


**3.** Build a database from a SQLite URL and from a PostgreSQL URL, print the class of each, and
show that neither is connected.


In [15]:
# your code here


**4.** Produce `database is locked` with two connections to one file, then show the same two writers
succeeding when the first transaction is closed before the second starts.


In [16]:
# your code here


**5.** Declare a model with a `TSVectorField` and show the statement that will not run on SQLite,
without running it.


In [17]:
# your code here


**6.** Take a connection from a pool, give it back, and print the pool's counts at each step.


In [18]:
# your code here


## Common errors

### peewee.OperationalError: database is locked


In [19]:
busy_path = str(WORK / "busy.db")
maker = SqliteDatabase(busy_path)
with maker.bind_ctx([Note]):
    maker.create_tables([Note])

holder = SqliteDatabase(busy_path, timeout=1)
waiter = SqliteDatabase(busy_path, timeout=1)
holder.connect()
waiter.connect()
holder.begin()
holder.execute_sql("INSERT INTO note (title, body, reads) VALUES (?, ?, ?)", ("held", "b", 0))

waiter.execute_sql("INSERT INTO note (title, body, reads) VALUES (?, ?, ?)", ("waiting", "b", 0))


OperationalError: database is locked

One writer at a time, and the second waited a second before giving up. The number of seconds is the
only thing between a wait and this exception, which is why it arrives under load and never in
testing.

Three things help, in this order: keep write transactions short, so the queue moves; raise the
timeout, so waiting is allowed; and set WAL, so that readers are not in the queue at all. None of
them makes two writers possible.


In [20]:
holder.rollback()                                                   # the transaction ends
print("once the first writer is done:",
      waiter.execute_sql("INSERT INTO note (title, body, reads) VALUES (?, ?, ?)",
                         ("waiting", "b", 0)).rowcount, "row written")
holder.close()
waiter.close()


once the first writer is done: 1 row written


True

### peewee.OperationalError: connection failed


In [21]:
nothing_there = connect_url("postgresql://someone@127.0.0.1:59999/catalog")
print("the object was built:", type(nothing_there).__name__)

try:
    nothing_there.connect()
except OperationalError as error:
    print("peewee.OperationalError:", refused(str(error)))


the object was built: PostgresqlDatabase
peewee.OperationalError: connection failed: connection to server at "127.0.0.1", port 59999 failed: ...


This is the boundary of this guide. peewee wraps whatever the driver said, and everything past this
line is PostgreSQL's own: where it is listening, what it will accept, and which user it believes
you are. **asyncpg and psycopg3, Deep Dive** is the guide that stands a server up and works through
these messages.

Note again where the failure was and was not. `connect_url` succeeded on a URL pointing at nothing,
because building a `Database` is not connecting to one. A settings mistake therefore surfaces at the
first query rather than at start up, unless something asks for a connection on purpose:


In [22]:
def checked(url):
    """Build the database and prove it answers, so a bad setting fails at start up."""
    database = connect_url(url)
    database.connect()
    database.close()
    return database


print("a good one:", type(checked(f"sqlite:///{WORK / 'catalog.db'}")).__name__)
try:
    checked("postgresql://someone@127.0.0.1:59999/catalog")
except OperationalError as error:
    print("a bad one:", refused(str(error)))


a good one: SqliteDatabase
a bad one: connection failed: connection to server at "127.0.0.1", port 59999 failed: ...


### peewee.OperationalError: near "USING": syntax error


In [23]:
class Ported(Model):
    body = TextField()
    search = TSVectorField()

    class Meta:
        database = sqlite


fresh = SqliteDatabase(str(WORK / "ported.db"))
with fresh.bind_ctx([Ported]):
    fresh.create_tables([Ported])


OperationalError: near "USING": syntax error

`TSVectorField` asks for a GIN index, and GIN is PostgreSQL's. The table would have been made: it is
the index beside it that has no meaning here, which is the same lesson as **Models and Fields**,
where an index turned out to be a statement of its own rather than part of the table.

There is no SQLite equivalent to switch to. The search that works here is the one in **FTS5Model and
SearchField**, and the two do not share an interface, so an application that must run on both writes
both and chooses:


In [24]:
def search_for(database):
    """Which search an application would use, decided by the backend it was given."""
    return "FTS5Model with SearchField" if isinstance(database, SqliteDatabase) else \
           "TSVectorField with a GIN index"


for url in (f"sqlite:///{WORK / 'catalog.db'}", "postgresql://user@db.internal:5432/catalog"):
    print(f"  {url.split('://')[0]:<11} -> {search_for(connect_url(url))}")


  sqlite      -> FTS5Model with SearchField
  postgresql  -> TSVectorField with a GIN index


### peewee.OperationalError: unrecognized token: "@"


In [25]:
with on_file.bind_ctx([Loose]):
    list(Loose.select().where(Loose.search.match("sea")))


OperationalError: unrecognized token: "@"

This is the same problem as the one above, arriving later and doing more damage. With the index
taken off, the table was created, rows were written and everything looked fine, because nothing had
searched yet. `match` compiles to `@@`, which SQLite cannot even tokenize.

A column of a type the database does not have is accepted by SQLite, stored, and read back. The
failure is not in the schema, it is in the first query that treats the column as what it was meant
to be:


In [26]:
with on_file.bind_ctx([Loose]):
    print("the column holds:", [(row.body, row.search) for row in Loose.select()])
    print("as plain text it is fine:",
          [row.body for row in Loose.select().where(Loose.search == "sea")])


the column holds: [('the sea', 'sea')]
as plain text it is fine: ['the sea']


## Recap

- The models do not name a backend. The `Database` object in scope decides the dialect, and
  `bind_ctx` puts a different one in scope for a block.
- A query compiles without a connection, so both backends' SQL can be printed side by side with
  nothing open.
- What changes: `INTEGER PRIMARY KEY` becomes `SERIAL`, `?` becomes `%s`, an insert gains a
  `RETURNING` clause, and `LIMIT` stops being a bound value.
- `connect_url` builds a `Database` from a URL and does not connect. A wrong URL fails at the first
  connection, not at the call, unless you ask for one on purpose.
- SQLite allows one writer. A second waits for `timeout` seconds and then raises `database is
  locked`. WAL stops readers from being blocked by a writer and does not allow a second writer.
- `TSVectorField` on SQLite fails twice: at the GIN index if there is one, and at `@@` on the first
  search if there is not.
- What the switch buys is mostly what earlier notebooks had to work around: enforced foreign keys,
  a transaction that aborts rather than half committing, and a pool that is worth having.
- A pooled database hands connections back on `close` and is emptied by `dispose`, which is what a
  forked child must call.


## What is next

The **A Small Catalog** notebook is the whole guide in one program: models, migrations, a loaded
catalog, search, and the report that makes the guide's claims about query counts checkable, ending
on what the backend switch would cost its ranking.


---

&#8592; **Previous:** [Migrations](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/peewee-deep-dive/10-migrations.ipynb)  &nbsp;·&nbsp;  [Peewee, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [A Small Catalog](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/peewee-deep-dive/12-a-small-catalog.ipynb) &#8594;
